### Improving reasoning-with inference time scaling

In [6]:
from pathlib import Path
import sys
import torch

ROOT_DIR = Path.cwd().parent  # Get parent of current directory
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))


from evaluating_reasoning_models.model_and_tokenizer import load_model_and_tokenizer

model, tokenizer = load_model_and_tokenizer(
    which_model="base",
    use_compile=False
)



In [7]:
from evaluating_reasoning_models.generate_text_streams import generate_text_stream_with_kv_cache
from evaluating_reasoning_models.render_prompt import render_prompt


In [8]:
raw_prompt = ("Half the value of $3x-9$ is $x+37$. "
    "What is the value of $x$?")

prompt = render_prompt(raw_prompt)

print(prompt)

You are a helpful math assistant.
Answer the question and write the final result on a new line as:
\boxed{ANSWER}

Question:
Half the value of $3x-9$ is $x+37$. What is the value of $x$?

Answer:


In [9]:
from evaluating_reasoning_models.complete_boxed import (
    has_complete_boxed_answer,
)


def generate_text_stream_concat_flex(
    model,
    tokenizer,
    prompt,
    device,
    max_new_tokens,
    verbose=False,
    generate_func=None,
    stop_after_boxed=True,
    stop_texts=("\nQuestion:", "\nAnswer:"),
    **generate_kwargs,
):
    if generate_func is None:
        generate_func = generate_text_stream_with_kv_cache

    generated_ids = []
    generated_text = ""

    for token in generate_func(
        prompt=prompt,
        model=model,
        tokenizer=tokenizer,
        device=device,
        max_new_tokens=max_new_tokens,
        eos_token_id=tokenizer.eos_token_id,
        **generate_kwargs,
    ):

        token_id = token.squeeze(0).item()
        generated_ids.append(token_id)

        token_text = tokenizer.decode([token_id])
        generated_text += token_text

        if verbose:
            print(
                token_text,
                end="",
                flush=True,
            )

        if stop_after_boxed and has_complete_boxed_answer(
            generated_text
        ):
            break

        if stop_texts and any(stop_text in generated_text for stop_text in stop_texts):
            break

    return generated_text

In [10]:
device = "cuda" if torch.cuda.is_available() else "cpu"
response = generate_text_stream_concat_flex(model, 
    tokenizer, 
    prompt, 
    device, 
    max_new_tokens=200, 
    verbose=True, 
    generate_func=generate_text_stream_with_kv_cache)

 \boxed{10}


### Now, using Chain-of-thought prompt ---for better response

In [11]:
prompt_cot = prompt + " \n\nExplain step step by step."

response = generate_text_stream_concat_flex(model, 
    tokenizer, 
    prompt_cot, 
    device, 
    max_new_tokens=200, 
    verbose=True)

 (Step 1: ...)

Step

 1: ... 

Step 2: ... 

Step 3: ... 

Step 4: ... 

Step 5: ... 

Step 6: ... 

Step 7: ... 

Step 8: ... 

Step 9: ... 

Step 10: ... 

Step 11: ... 

Step 12: ... 

Step 13: ... 

Step 14: ... 

Step 15: ... 

Step 16: ... 

Step 17: ... 

Step 18: ... 

Step 19: ... 

Step 20: ... 

Step 21: ... 

Step 22: ... 

Step 23: ... 

Step 24: ... 

Step 25: ... 

Step 26: ... 

Step 27: ... 

Step 28: ... 

Step 29: ...

### Controlling output diversity with temperature scaling
* starting with temperature scaling

* ok, now  ---> process of selecting the token

In [12]:
prompt_example = "The capital of germany is, give answer in single word"

response = generate_text_stream_concat_flex(model, 
    tokenizer, 
    prompt_example, 
    device, 
    max_new_tokens=200, 
    verbose=True)

??
?
A. Berlin
B.

 Berlin
C. Berlin
D. Berlin

Wait, but the question says "give answer in single word", but all options are the same. Maybe there's a typo? Or maybe the answer is just Berlin. Let me check again. Yes, the capital of Germany is Berlin. So the correct answer is A. Berlin.
The answer is A. Berlin.
**Final Answer**
A. Berlin
**Final Answer**
A. Berlin
**Final Answer**
A. Berlin
**Final Answer**
A. Berlin
**Final Answer**
A. Berlin
**Final Answer**
A. Berlin
**Final Answer**
A. Berlin
**Final Answer**
A. Berlin
**Final Answer**
A. Berlin
**Final Answer**
A. Berlin
**Final Answer**
A. Berlin
**Final Answer**
A. Berlin
**Final Answer**
A. Berlin
**Final Answer**
A. Berlin
**Final Answer**
A

In [13]:
## not  working properly